In [0]:
# ── Imports ──────────────────────────────────────────────────────────────
# F: funciones de columnas de Spark (equivalentes a funciones SQL: current_timestamp, col, lit...)
# Config y audit_run: nuestras librerías de lib/
from pyspark.sql import functions as F
from lib.config import Config
from lib.audit import audit_run

# ── Parámetros del notebook (widgets) ────────────────────────────────────
# Equivalen a la "parameter cell" de Fabric. Un Job de Databricks o ADF
# (baseParameters) los rellenan al lanzar el notebook. Si se ejecuta a mano,
# toman el valor por defecto.
#   catalog    → entorno (workspace, dev, prod): mismo código para todos
#   job_run_id → id de la ejecución del orquestador, para cruzarlo con nuestro log
dbutils.widgets.text("catalog", "workspace", "Catálogo")
dbutils.widgets.text("job_run_id", "", "Id de ejecución del orquestador")

cfg = Config(catalog=dbutils.widgets.get("catalog"))
JOB_RUN_ID = dbutils.widgets.get("job_run_id")

# ── Rutas y nombres ──────────────────────────────────────────────────────
# SOURCE:     carpeta donde llegan los JSON
# TARGET:     tabla bronze destino (catalogo.bronze.transactions)
# CHECKPOINT: donde Auto Loader recuerda qué archivos ya procesó
SOURCE     = f"{cfg.landing_path}/transactions"
TARGET     = cfg.table("bronze", "transactions")
CHECKPOINT = f"{cfg.checkpoint_path}/bronze_transactions"

In [0]:
# ── Ejecución auditada ───────────────────────────────────────────────────
# Todo lo que está dentro del "with" queda registrado en ops.pipeline_runs:
# una fila RUNNING al empezar y SUCCESS/FAILED al terminar (con el error si falla).
with audit_run(cfg, "02_bronze", "bronze", TARGET, JOB_RUN_ID) as run:

    # ── Lectura incremental con Auto Loader ──────────────────────────────
    # Lee SOLO los archivos JSON que todavía no procesó (lo sabe por el checkpoint).
    #   schemaLocation:     guarda el schema inferido y sus versiones
    #   inferColumnTypes:   detecta números y fechas (si no, todo entra como texto)
    #   addNewColumns:      si el origen agrega una columna, la incorpora a la tabla
    #   _rescued_data:      columna automática con los valores que no encajan en el schema
    # El select agrega los campos de auditoría de bronze:
    #   _ingested_at:   cuándo se cargó la fila
    #   _source_file:   de qué archivo vino
    #   _source_file_ts: fecha del archivo en el origen
    #   _run_id:        qué ejecución la cargó (enlaza con ops.pipeline_runs)
    df = (spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", "json")
          .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/_schema")
          .option("cloudFiles.inferColumnTypes", "true")
          # schemaHints: fija el tipo de las columnas críticas (el resto se infiere).
          # Sin esto, el tipo depende de qué archivos existan en la primera carga.
          .option("cloudFiles.schemaHints",
                  "transactionID BIGINT, customerID BIGINT, franchiseID BIGINT, "
                  "quantity BIGINT, dateTime TIMESTAMP, "
                  "createdAt TIMESTAMP, updatedAt TIMESTAMP")
          .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
          .load(SOURCE)
          .select("*",
                  F.current_timestamp().alias("_ingested_at"),
                  F.col("_metadata.file_path").alias("_source_file"),
                  F.col("_metadata.file_modification_time").alias("_source_file_ts"),
                  F.lit(run.run_id).alias("_run_id")))

    # ── Escritura en la tabla bronze ─────────────────────────────────────
    # Escribe las filas nuevas en la tabla Delta (la crea si no existe).
    #   checkpointLocation: registra qué se procesó (garantiza exactly-once)
    #   mergeSchema:        permite que la tabla acepte columnas nuevas
    #   availableNow:       procesa todo lo pendiente y termina (modo batch)
    # awaitTermination espera a que termine antes de seguir.
    query = (df.writeStream
             .option("checkpointLocation", CHECKPOINT)
             .option("mergeSchema", "true")
             .trigger(availableNow=True)
             .toTable(TARGET))
    query.awaitTermination()

    # ── Métricas para el log de auditoría ────────────────────────────────
    # Cuenta las filas que cargó ESTA ejecución (filtrando por su _run_id)
    # y cuántas traen datos rescatados (valores con tipo o formato inesperado).
    nuevas = spark.table(TARGET).filter(F.col("_run_id") == run.run_id)
    run.rows_read     = nuevas.count()
    run.rows_written  = run.rows_read
    run.rows_rejected = nuevas.filter(F.col("_rescued_data").isNotNull()).count()

print(f"Filas nuevas: {run.rows_written} | con datos rescatados: {run.rows_rejected}")

In [0]:
%sql
SELECT _source_file, _run_id, COUNT(*) AS filas
FROM workspace.bronze.transactions
GROUP BY ALL;

In [0]:
%sql
-- Historial de ejecuciones de bronze, desde la vista de auditoría
-- (la vista devuelve el estado final de cada ejecución: SUCCESS o FAILED)
SELECT task, status, rows_read, rows_written, rows_rejected, started_at
FROM workspace.ops.v_pipeline_runs
WHERE task = '02_bronze'
ORDER BY started_at;

In [0]:
%sql
-- Inventario de problemas que hay en bronze (todo se resuelve en silver):
--   1) transacciones repetidas → las correcciones del lote 3
--   2) cantidad <= 0           → filas inválidas del lote 4
--   3) cliente nulo            → filas inválidas del lote 4
--   4) _rescued_data no nulo   → valores que no encajaron en el schema (quantity = "dos")
SELECT 'transactionID repetido' AS problema, COUNT(*) AS casos
FROM (SELECT transactionID
      FROM workspace.bronze.transactions
      GROUP BY transactionID
      HAVING COUNT(*) > 1)
UNION ALL
SELECT 'quantity <= 0', COUNT(*)
FROM workspace.bronze.transactions WHERE quantity <= 0
UNION ALL
SELECT 'customerID nulo', COUNT(*)
FROM workspace.bronze.transactions WHERE customerID IS NULL
UNION ALL
SELECT 'con _rescued_data', COUNT(*)
FROM workspace.bronze.transactions WHERE _rescued_data IS NOT NULL;

In [0]:
%sql
-- La fila con quantity = "dos": el campo quantity queda NULL
-- y el valor original se guarda en _rescued_data (no se pierde).
SELECT transactionID, product, quantity, _rescued_data
FROM workspace.bronze.transactions
WHERE _rescued_data IS NOT NULL;